# Back scratcher - DOFBOT SE

**How to use:** run cells 1, 2 and 3 once (Shift+Enter on each). Then run only the cell you need.

**STOP:** press the **■ Stop** button in the toolbar (or press `I` twice). The arm lifts off and parks.  
Keep the arm's power plug within reach as the real emergency stop.

Close Yahboom's app first, in a terminal: `cd ~/rosmaster && ./kill_YahboomArm.sh`

## 1. Settings
Change numbers here, then run this cell again. `SIM = True` = test without moving the arm.

In [ ]:
import json, os, random, sys, time

SIM = False   # True = only print moves, arm doesn't move

POINTS_FILE = os.path.expanduser("~/scratch_points.json")

# ----------------------------------------------------------------------------


# Pose the arm rests in when not scratching (folded, away from your body).
# Record your own with "read" mode and paste it here.
PARK_POSE = [90, 150, 10, 10, 90, 90]

# Hard safety limits per servo [min, max]. The arm will never be commanded
# outside these. Narrow them so the arm physically CANNOT reach your head/neck.
# Servo 1 = base rotation, 2 = shoulder, 3 = elbow, 4 = wrist,
# 5 = wrist rotation (0-270), 6 = gripper.
LIMITS = {
    1: (20, 160),
    2: (40, 160),
    3: (0, 120),
    4: (0, 150),
    5: (0, 270),
    6: (60, 180),
}

GRIP_ANGLE = 150          # gripper closed on the brush/scratcher handle

# Lifting off the back: which servo moves the brush AWAY from you, and how much.
# Sign matters! Test without lying down: "+" or "-" must move it away from the
# mattress / your body side.
LIFT_SERVO = 2
LIFT_DEG = +12

# The scratching motion: small back-and-forth on one servo around each spot.
SCRATCH_SERVO = 1         # 1 = side-to-side strokes, 4 = up-and-down strokes
SCRATCH_AMPLITUDE = 8     # degrees each way (keep small: 4-12)
STROKES_PER_SPOT = 8
STROKE_MS = 350           # time per stroke in ms (bigger = slower, gentler)

MOVE_MS = 1200            # time for normal moves between spots (keep slow)
MIN_MOVE_MS = 250         # never move faster than this

DEFAULT_MINUTES = 5       # auto-stop after this long


## 2. Program (just run it, no need to change)

In [ ]:
# ----------------------------------------------------------------------------

class SimArm:
    """Stand-in so you can test the logic in the VM without the robot."""

    def __init__(self):
        self.angles = list(PARK_POSE)

    def Arm_serial_servo_write6_array(self, a, t):
        self.angles = list(a)
        print(f"  [sim] move to {a} in {t} ms")

    def Arm_serial_servo_write(self, sid, angle, t):
        self.angles[sid - 1] = angle
        print(f"  [sim] servo {sid} -> {angle} in {t} ms")

    def Arm_serial_servo_read(self, sid):
        return self.angles[sid - 1]

    def Arm_serial_set_torque(self, on):
        print(f"  [sim] torque {'ON' if on else 'OFF'}")


def connect(sim):
    if sim:
        return SimArm()
    try:
        from Arm_Lib import Arm_Device
    except ImportError:
        sys.exit("Arm_Lib not found. Run this inside Yahboom's DOFBOT SE VM image, "
                 "or use --sim to test without the arm.")
    arm = Arm_Device()
    time.sleep(0.1)
    return arm


def clamp_pose(pose):
    out = []
    for i, a in enumerate(pose, start=1):
        lo, hi = LIMITS[i]
        out.append(int(max(lo, min(hi, a))))
    return out


def move(arm, pose, ms=MOVE_MS, sim=False):
    pose = clamp_pose(pose)
    ms = max(MIN_MOVE_MS, int(ms))
    arm.Arm_serial_servo_write6_array(pose, ms)
    time.sleep(0 if sim else ms / 1000 + 0.05)
    return pose


_last_pose = list(PARK_POSE)

def read_pose(arm):
    """Read all 6 servos. A failed read keeps that servo's last known value
    instead of crashing (bus reads often fail while you move the arm by hand).
    The gripper (6) is never read - it just holds GRIP_ANGLE."""
    global _last_pose
    pose = list(_last_pose)
    for sid in range(1, 6):
        val = None
        for _ in range(10):  # retry
            try:
                val = arm.Arm_serial_servo_read(sid)
            except Exception:
                val = None
            hi = 270 if sid == 5 else 180
            if val is not None and 0 <= val <= hi:
                break
            val = None
            time.sleep(0.03)
        if val is None:
            print(f"  (servo {sid}: no reading, keeping {pose[sid - 1]})")
        else:
            pose[sid - 1] = int(val)
    pose[5] = GRIP_ANGLE
    _last_pose = pose
    return list(pose)


def lifted(pose):
    p = list(pose)
    p[LIFT_SERVO - 1] += LIFT_DEG
    return p


def park(arm, sim):
    print("Parking...")
    try:
        current = read_pose(arm)
        move(arm, lifted(current), 800, sim)   # lift off the back first
    except Exception:
        pass
    move(arm, PARK_POSE, 2000, sim)


# ----------------------------------------------------------------------------
# Modes
# ----------------------------------------------------------------------------

def mode_read(arm, sim):
    arm.Arm_serial_set_torque(0)
    print("Torque OFF - move the arm by hand. Ctrl+C to stop.")
    try:
        while True:
            print(read_pose(arm))
            time.sleep(0.5)
    except KeyboardInterrupt:
        arm.Arm_serial_set_torque(1)
        print("\nTorque ON")


def mode_teach(arm, sim):
    print("TEACH MODE")
    print("Tip: put a pillow / rolled blanket where your back will be and teach")
    print("on that. Position the brush so the bristles JUST touch - not pressed in.\n")
    move(arm, PARK_POSE, 2000, sim)
    arm.Arm_serial_set_torque(0)
    points = []
    try:
        while True:
            cmd = input(f"[{len(points)} spots] Move arm to a spot, Enter = save, "
                        f"u = undo, q = finish: ").strip().lower()
            if cmd == "q":
                break
            if cmd == "u":
                if points:
                    print("  removed", points.pop())
                continue
            pose = read_pose(arm)
            pose[5] = GRIP_ANGLE  # keep the brush gripped
            pose = clamp_pose(pose)
            points.append(pose)
            print("  saved", pose)
    finally:
        arm.Arm_serial_set_torque(1)
    if points:
        with open(POINTS_FILE, "w") as f:
            json.dump(points, f, indent=2)
        print(f"Saved {len(points)} spots to {POINTS_FILE}")
    park(arm, sim)


def scratch_spot(arm, spot, sim):
    up = lifted(spot)
    move(arm, up, MOVE_MS, sim)         # hover above the spot
    move(arm, spot, 600, sim)           # gently touch down
    idx = SCRATCH_SERVO - 1
    for i in range(STROKES_PER_SPOT):
        p = list(spot)
        p[idx] += SCRATCH_AMPLITUDE if i % 2 == 0 else -SCRATCH_AMPLITUDE
        move(arm, p, STROKE_MS, sim)
    move(arm, spot, STROKE_MS, sim)
    move(arm, up, 500, sim)             # lift off before moving on


def mode_run(arm, sim, minutes, shuffle):
    if not os.path.exists(POINTS_FILE):
        sys.exit("No spots recorded yet. Run:  python3 backscratch.py teach")
    with open(POINTS_FILE) as f:
        points = [clamp_pose(p) for p in json.load(f)]
    print(f"Scratching {len(points)} spots for up to {minutes} min. Ctrl+C = stop.")
    for s in (3, 2, 1):
        print(f"  starting in {s}...")
        time.sleep(0 if sim else 1)
    deadline = time.time() + minutes * 60
    rounds = 0
    try:
        while time.time() < deadline:
            order = points[:]
            if shuffle:
                random.shuffle(order)
            for spot in order:
                if time.time() >= deadline:
                    break
                scratch_spot(arm, spot, sim)
            rounds += 1
            if sim and rounds >= 1:
                break
        print("Time's up.")
    except KeyboardInterrupt:
        print("\nStopped by you.")
    finally:
        park(arm, sim)


## 3. Connect to the arm

In [ ]:
arm = connect(SIM)
print('Connected' if not SIM else 'Simulation mode')

## Read angles
Motors go loose; move the arm by hand and watch the numbers. Press ■ Stop when done.
Use this to find a good folded `PARK_POSE`, then put it in cell 1.

In [ ]:
mode_read(arm, SIM)

## Teach scratch spots
Pillow on the bed. Move the arm by hand so the brush *just* touches it.
Type in the box that appears: **Enter** = save spot, **u** = undo, **q** = finish.

In [ ]:
mode_teach(arm, SIM)

## Scratch!
Change `minutes` and `shuffle` (random order) if you want. ■ Stop = stop and park.

In [ ]:
mode_run(arm, SIM, minutes=5, shuffle=False)

## Park (fold away)

In [ ]:
park(arm, SIM)